# Rossmann Store Sales — Baseline Forecasting

## 1. Problem Definition

The goal is to predict daily sales for each Rossmann store.

Each observation represents one store on one date.

**Target Variable:** Sales

### Forecasting Assumptions

- Store opening status (Open) is assumed to be known.
- Closed stores have predicted sales equal to zero.
- Customers will not be used because future customer counts are not assumed to be available.

### Training and Validation Strategy

- Training period: 2013-01-01 to 2015-06-30
- Validation period: 2015-07-01 to 2015-07-31

Validation data must not be used to calculate historical sales statistics for the baseline.

### Initial Baseline

Predict sales using each store's historical average daily sales from the training period.

### Evaluation Metrics

- MAE (Mean Absolute Error)
- RMSE (Root Mean Squared Error)

The baseline will serve as a reference for future models.

In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "train_clean.parquet"

df = pd.read_parquet(DATA_PATH)

print("Shape:", df.shape)
print("Date range:", df["Date"].min(), "to", df["Date"].max())

df.head()

In [ ]:
split_date = pd.Timestamp("2015-07-01")

train_data = df.loc[df["Date"] < split_date].copy()

valid_data = df.loc[df["Date"] >= split_date].copy()

print("Training shape:", train_data.shape)
print("Validation shape:", valid_data.shape)

print("Training end:", train_data["Date"].max())
print("Validation start:", valid_data["Date"].min())
print("Validation end:", valid_data["Date"].max())

## 2. Historical Store Mean Baseline

Calculate the average sales of each store using only open days from the training period.

These historical averages will be used to predict sales for open stores in the validation period.

In [ ]:
train_open = train_data.loc[train_data["Open"] == 1].copy()

store_mean_sales = train_open.groupby("Store")["Sales"].mean()

print("Number of stores:", len(store_mean_sales))

store_mean_sales.head()

In [ ]:
valid_predictions = valid_data[["Store", "Date", "Open", "Sales"]].copy()

valid_predictions["PredictedSales"] = valid_predictions["Store"].map(store_mean_sales)

print(
    "Missing predictions:",
    valid_predictions["PredictedSales"].isna().sum(),
)

valid_predictions.head()

In [ ]:
valid_predictions.loc[
    valid_predictions["Open"] == 0,
    "PredictedSales",
] = 0

valid_predictions[["Store", "Date", "Open", "Sales", "PredictedSales"]].head(10)

In [ ]:
closed_valid = valid_predictions.loc[valid_predictions["Open"] == 0]

print("Closed observations:", len(closed_valid))

print(
    "Non-zero predictions for closed stores:",
    (closed_valid["PredictedSales"] != 0).sum(),
)

## 3. Baseline Evaluation

Evaluate the historical store mean baseline on the July 2015 validation period using MAE and RMSE.

In [ ]:
errors = valid_predictions["Sales"] - valid_predictions["PredictedSales"]

mae = errors.abs().mean()
rmse = (errors.pow(2).mean()) ** 0.5

print(f"Validation MAE: {mae:.2f}")
print(f"Validation RMSE: {rmse:.2f}")

In [ ]:
closed_mask = valid_predictions["Open"] == 0

print(
    "Non-zero predictions on closed days:",
    (valid_predictions.loc[closed_mask, "PredictedSales"] != 0).sum(),
)

valid_open = valid_predictions.loc[valid_predictions["Open"] == 1].copy()

open_errors = valid_open["Sales"] - valid_open["PredictedSales"]

open_mae = open_errors.abs().mean()
open_rmse = open_errors.pow(2).mean() ** 0.5

print("Open-day observations:", len(valid_open))
print(f"Open-day MAE: {open_mae:.2f}")
print(f"Open-day RMSE: {open_rmse:.2f}")

### Baseline Evaluation — Findings

The historical store mean baseline achieved a validation MAE of 1,155.40 and RMSE of 1,620.72 across all observations.

For open-store observations only, MAE increased to 1,322.92 and RMSE to 1,734.24.

The difference occurs because closed stores have zero actual and predicted sales, reducing the overall error metrics.

Both overall and open-day metrics will be used to evaluate future baseline improvements.

## 4. Store and Day-of-Week Baseline

Improve the initial baseline by calculating historical average sales for each Store and DayOfWeek combination using training data only.

In [ ]:
store_weekday_stats = train_open.groupby(["Store", "DayOfWeek"])["Sales"].agg(
    ["count", "mean"]
)

store_weekday_stats.loc[1]

In [ ]:
weekday_valid = valid_data[["Store", "Date", "DayOfWeek", "Open", "Sales"]].copy()

weekday_means = store_weekday_stats["mean"].reset_index(name="WeekdayMeanSales")

weekday_predictions = weekday_valid.merge(
    weekday_means,
    on=["Store", "DayOfWeek"],
    how="left",
    validate="many_to_one",
)

weekday_predictions.head()

In [ ]:
print(
    "Missing weekday means:",
    weekday_predictions["WeekdayMeanSales"].isna().sum(),
)

weekday_predictions["PredictedSales"] = weekday_predictions["WeekdayMeanSales"].fillna(
    weekday_predictions["Store"].map(store_mean_sales)
)

weekday_predictions.loc[
    weekday_predictions["Open"] == 0,
    "PredictedSales",
] = 0

print(
    "Missing final predictions:",
    weekday_predictions["PredictedSales"].isna().sum(),
)

print(
    "Non-zero predictions on closed days:",
    (
        weekday_predictions.loc[weekday_predictions["Open"] == 0, "PredictedSales"] != 0
    ).sum(),
)

In [ ]:
missing_open_weekday = (
    weekday_predictions.loc[
        weekday_predictions["Open"] == 1,
        "WeekdayMeanSales",
    ]
    .isna()
    .sum()
)

print("Missing weekday means on open days:", missing_open_weekday)

In [ ]:
weekday_errors = weekday_predictions["Sales"] - weekday_predictions["PredictedSales"]

weekday_mae = weekday_errors.abs().mean()
weekday_rmse = weekday_errors.pow(2).mean() ** 0.5

weekday_open_errors = weekday_errors.loc[weekday_predictions["Open"] == 1]

weekday_open_mae = weekday_open_errors.abs().mean()
weekday_open_rmse = weekday_open_errors.pow(2).mean() ** 0.5

print(f"Overall MAE: {weekday_mae:.2f}")
print(f"Overall RMSE: {weekday_rmse:.2f}")
print(f"Open-day MAE: {weekday_open_mae:.2f}")
print(f"Open-day RMSE: {weekday_open_rmse:.2f}")

### Store and Day-of-Week Baseline — Findings

The Store + DayOfWeek baseline outperformed the historical Store Mean baseline on July 2015 validation data.

- Overall MAE decreased from 1,155.40 to 1,002.94 (approximately 13.2% improvement).
- Overall RMSE decreased from 1,620.72 to 1,388.01 (approximately 14.4% improvement).
- Open-day MAE decreased from 1,322.92 to 1,148.36.
- Open-day RMSE decreased from 1,734.24 to 1,485.23.

All open-day validation observations had available historical Store + DayOfWeek averages, so no fallback was required for open stores.

These results show that incorporating weekly sales patterns improves forecasting accuracy compared with using a single historical average per store.

However, performance has only been evaluated on one validation month.

In [ ]:
weekday_predictions["AbsoluteError"] = (
    weekday_predictions["Sales"] - weekday_predictions["PredictedSales"]
).abs()

In [ ]:
weekday_error_by_day = (
    weekday_predictions.loc[weekday_predictions["Open"] == 1]
    .groupby("DayOfWeek")["AbsoluteError"]
    .agg(["count", "mean", "median"])
)

weekday_error_by_day

In [ ]:
weekday_predictions["SignedError"] = (
    weekday_predictions["Sales"] - weekday_predictions["PredictedSales"]
)

weekday_bias_by_day = (
    weekday_predictions.loc[weekday_predictions["Open"] == 1]
    .groupby("DayOfWeek")["SignedError"]
    .agg(["count", "mean", "median"])
)

weekday_bias_by_day

## 5. Promotion-Based Error Analysis

Investigate whether the Store + DayOfWeek baseline produces different prediction errors on promotional and non-promotional days.

In [ ]:
promo_error_data = weekday_predictions.merge(
    valid_data[["Store", "Date", "Promo"]],
    on=["Store", "Date"],
    how="left",
    validate="one_to_one",
)

promo_error_summary = (
    promo_error_data.loc[promo_error_data["Open"] == 1]
    .groupby("Promo")
    .agg(
        observation_count=("Sales", "size"),
        mae=("AbsoluteError", "mean"),
        mean_signed_error=("SignedError", "mean"),
    )
)

promo_error_summary

## 6. Store, Day-of-Week, and Promotion Baseline

The previous baseline tends to overpredict sales on non-promotional days and underpredict sales on promotional days.

We will calculate historical average sales for each Store, DayOfWeek, and Promo combination using training data only.

In [ ]:
store_weekday_promo_stats = train_open.groupby(["Store", "DayOfWeek", "Promo"])[
    "Sales"
].agg(["count", "mean"])

store_weekday_promo_stats.loc[1]

In [ ]:
promo_means = store_weekday_promo_stats["mean"].reset_index(name="PromoMeanSales")

promo_valid = valid_data[
    ["Store", "Date", "DayOfWeek", "Promo", "Open", "Sales"]
].copy()

promo_predictions = promo_valid.merge(
    promo_means,
    on=["Store", "DayOfWeek", "Promo"],
    how="left",
    validate="many_to_one",
)

promo_predictions.head()

In [ ]:
print(
    "Missing promo means:",
    promo_predictions["PromoMeanSales"].isna().sum(),
)

print(
    "Missing promo means on open days:",
    promo_predictions.loc[
        promo_predictions["Open"] == 1,
        "PromoMeanSales",
    ]
    .isna()
    .sum(),
)

In [ ]:
promo_predictions["PredictedSales"] = promo_predictions["PromoMeanSales"].copy()

promo_predictions.loc[
    promo_predictions["Open"] == 0,
    "PredictedSales",
] = 0

print(
    "Missing final predictions:",
    promo_predictions["PredictedSales"].isna().sum(),
)

print(
    "Non-zero predictions on closed days:",
    (
        promo_predictions.loc[
            promo_predictions["Open"] == 0,
            "PredictedSales",
        ]
        != 0
    ).sum(),
)

In [ ]:
promo_errors = promo_predictions["Sales"] - promo_predictions["PredictedSales"]

promo_mae = promo_errors.abs().mean()
promo_rmse = promo_errors.pow(2).mean() ** 0.5

promo_open_errors = promo_errors.loc[promo_predictions["Open"] == 1]

promo_open_mae = promo_open_errors.abs().mean()
promo_open_rmse = promo_open_errors.pow(2).mean() ** 0.5

print(f"Overall MAE: {promo_mae:.2f}")
print(f"Overall RMSE: {promo_rmse:.2f}")
print(f"Open-day MAE: {promo_open_mae:.2f}")
print(f"Open-day RMSE: {promo_open_rmse:.2f}")

### Store + DayOfWeek + Promo Baseline — Findings

The third baseline achieved the best performance among the three evaluated approaches.

- Overall MAE: 639.46
- Overall RMSE: 970.52
- Open-day MAE: 732.18
- Open-day RMSE: 1,038.49

Compared with the Store + DayOfWeek baseline, open-day MAE decreased by approximately 36.2%, while RMSE decreased by approximately 30.1%.

This improvement indicates that regular promotion status provides valuable predictive information beyond store identity and day of week.

The baseline uses historical training statistics only and assumes that future Open and Promo values are known.

Results have been evaluated on July 2015 only and require further temporal validation.

In [ ]:
promo_predictions["AbsoluteError"] = promo_errors.abs()

promo_predictions["SignedError"] = promo_errors

promo3_error_by_promo = (
    promo_predictions.loc[promo_predictions["Open"] == 1]
    .groupby("Promo")
    .agg(
        observation_count=("Sales", "size"),
        mae=("AbsoluteError", "mean"),
        mean_signed_error=("SignedError", "mean"),
    )
)

promo3_error_by_promo

### Promotion-Based Error Analysis — Baseline 3

Adding Promo substantially improved prediction accuracy and reduced systematic prediction bias.

For non-promotional days:
- MAE decreased from 1,047.68 to 640.75.
- Mean signed error changed from -821.60 to +56.57.

For promotional days:
- MAE decreased from 1,257.71 to 831.48.
- Mean signed error decreased from +1,114.83 to +127.44.

The third baseline provides more balanced predictions for both promotional and non-promotional days.

However, promotional days still have higher prediction errors, and the model has only been evaluated on July 2015.

## 7. Temporal Validation

In [ ]:
validation_months = pd.to_datetime(
    ["2015-04-01", "2015-05-01", "2015-06-01", "2015-07-01"]
)

for start_date in validation_months:
    end_date = start_date + pd.offsets.MonthBegin(1)

    fold_train = df.loc[df["Date"] < start_date]

    fold_valid = df.loc[(df["Date"] >= start_date) & (df["Date"] < end_date)]

    print(
        f"{start_date:%Y-%m}: "
        f"Train = {len(fold_train):,}, "
        f"Validation = {len(fold_valid):,}"
    )

In [ ]:
april_start = pd.Timestamp("2015-04-01")
april_end = pd.Timestamp("2015-05-01")

april_train = df.loc[df["Date"] < april_start].copy()

april_valid = df.loc[(df["Date"] >= april_start) & (df["Date"] < april_end)].copy()

april_train_open = april_train.loc[april_train["Open"] == 1]

In [ ]:
april_means = (
    april_train_open.groupby(["Store", "DayOfWeek", "Promo"])["Sales"]
    .mean()
    .reset_index(name="HistoricalMeanSales")
)

april_means.head()

In [ ]:
april_predictions = april_valid[
    ["Store", "Date", "DayOfWeek", "Promo", "Open", "Sales"]
].merge(
    april_means,
    on=["Store", "DayOfWeek", "Promo"],
    how="left",
    validate="many_to_one",
)

print("Validation rows:", len(april_predictions))

print(
    "Missing means on open days:",
    april_predictions.loc[
        april_predictions["Open"] == 1,
        "HistoricalMeanSales",
    ]
    .isna()
    .sum(),
)

april_predictions.head()

In [ ]:
april_predictions["PredictedSales"] = april_predictions["HistoricalMeanSales"].copy()

april_predictions.loc[
    april_predictions["Open"] == 0,
    "PredictedSales",
] = 0

print(
    "Missing final predictions:",
    april_predictions["PredictedSales"].isna().sum(),
)

print(
    "Non-zero predictions on closed days:",
    (
        april_predictions.loc[
            april_predictions["Open"] == 0,
            "PredictedSales",
        ]
        != 0
    ).sum(),
)

In [ ]:
april_errors = april_predictions["Sales"] - april_predictions["PredictedSales"]

april_mae = april_errors.abs().mean()
april_rmse = april_errors.pow(2).mean() ** 0.5

april_open_errors = april_errors.loc[april_predictions["Open"] == 1]

april_open_mae = april_open_errors.abs().mean()
april_open_rmse = april_open_errors.pow(2).mean() ** 0.5

print(f"April Overall MAE: {april_mae:.2f}")
print(f"April Overall RMSE: {april_rmse:.2f}")
print(f"April Open-day MAE: {april_open_mae:.2f}")
print(f"April Open-day RMSE: {april_open_rmse:.2f}")

### April 2015 Validation Findings

The Store + DayOfWeek + Promo baseline achieved:

- Overall MAE: 783.09
- Overall RMSE: 1,332.09
- Open-day MAE: 972.65
- Open-day RMSE: 1,484.59

All open-day observations had available historical group averages, and closed-day predictions were correctly set to zero.

Errors were higher than those observed in July 2015. However, this difference may reflect changes in training history, seasonal conditions, or sales patterns.

Additional validation months are needed to assess model stability.

In [ ]:
def evaluate_promo_baseline(train_df, valid_df):
    train_open = train_df.loc[train_df["Open"] == 1]

    historical_means = (
        train_open.groupby(["Store", "DayOfWeek", "Promo"])["Sales"]
        .mean()
        .reset_index(name="HistoricalMeanSales")
    )

    predictions = valid_df[["Store", "DayOfWeek", "Promo", "Open", "Sales"]].merge(
        historical_means,
        on=["Store", "DayOfWeek", "Promo"],
        how="left",
        validate="many_to_one",
    )

    missing_open = (
        predictions.loc[
            predictions["Open"] == 1,
            "HistoricalMeanSales",
        ]
        .isna()
        .sum()
    )

    if missing_open > 0:
        raise ValueError(
            f"Missing historical means for {missing_open} open observations"
        )

    predictions["PredictedSales"] = predictions["HistoricalMeanSales"]

    predictions.loc[
        predictions["Open"] == 0,
        "PredictedSales",
    ] = 0

    errors = predictions["Sales"] - predictions["PredictedSales"]
    open_errors = errors.loc[predictions["Open"] == 1]

    return {
        "overall_mae": errors.abs().mean(),
        "overall_rmse": errors.pow(2).mean() ** 0.5,
        "open_mae": open_errors.abs().mean(),
        "open_rmse": open_errors.pow(2).mean() ** 0.5,
    }

In [ ]:
april_metrics = evaluate_promo_baseline(
    april_train,
    april_valid,
)

for metric, value in april_metrics.items():
    print(f"{metric}: {value:.2f}")

In [ ]:
fold_results = []

for start_date in validation_months:
    end_date = start_date + pd.offsets.MonthBegin(1)

    fold_train = df.loc[df["Date"] < start_date].copy()

    fold_valid = df.loc[(df["Date"] >= start_date) & (df["Date"] < end_date)].copy()

    metrics = evaluate_promo_baseline(
        fold_train,
        fold_valid,
    )

    metrics["month"] = start_date.strftime("%Y-%m")

    fold_results.append(metrics)

results_df = pd.DataFrame(fold_results).set_index("month")

results_df.round(2)

### Temporal Validation — Findings

The Store + DayOfWeek + Promo baseline was evaluated using four expanding-window validation folds from April to July 2015.

Open-day MAE decreased from 972.65 in April to 732.18 in July, while open-day RMSE decreased from 1,484.59 to 1,038.49.

April showed the highest errors, suggesting that forecasting difficulty varies across validation periods.

The improvement across months may reflect changes in training history, seasonal patterns, or other business conditions.

Further error analysis is needed to understand the larger errors observed in April.

These results evaluate the third baseline across multiple months but do not yet establish its superiority over the other baselines in every month.

In [ ]:
april_open_eval = april_predictions.loc[april_predictions["Open"] == 1].copy()

april_open_eval["AbsoluteError"] = (
    april_open_eval["Sales"] - april_open_eval["PredictedSales"]
).abs()

april_error_by_date = (
    april_open_eval.groupby("Date")
    .agg(
        open_store_count=("Store", "size"),
        daily_mae=("AbsoluteError", "mean"),
    )
    .sort_values("daily_mae", ascending=False)
)

april_error_by_date.head(10)

In [ ]:
dates_to_check = pd.to_datetime(
    [
        "2015-04-01",
        "2015-04-02",
        "2015-04-03",
        "2015-04-04",
        "2015-04-06",
        "2015-04-30",
    ]
)

april_holiday_check = (
    april_valid.loc[april_valid["Date"].isin(dates_to_check)]
    .groupby(["Date", "StateHoliday"])
    .agg(
        observation_count=("Store", "size"),
        open_store_count=("Open", "sum"),
    )
)

april_holiday_check

In [ ]:
april_error_by_date["total_absolute_error"] = (
    april_error_by_date["daily_mae"] * april_error_by_date["open_store_count"]
)

april_error_by_date["error_share_pct"] = (
    april_error_by_date["total_absolute_error"]
    / april_error_by_date["total_absolute_error"].sum()
    * 100
)

april_error_by_date.sort_values(
    "total_absolute_error",
    ascending=False,
).head(10).round(2)

### April Error Analysis — Final Findings

The April 2015 analysis revealed substantial differences in prediction errors across dates.

April 3 and April 6 had very high daily MAE values, but relatively few stores were open due to Easter holidays.

In contrast, April 1, 2, 4, and 30 collectively accounted for approximately 44.4% of the month's total absolute prediction error.

This demonstrates that dates with the highest daily MAE do not necessarily contribute the most to overall model error.

The results suggest that calendar-related features, including proximity to holidays, may be useful for future forecasting models.

**Decision:** Retain all observations and investigate calendar-related patterns in future models rather than modifying or removing difficult validation dates.

## 8. Baseline Comparison Across Validation Folds

Compare the Store Mean, Store + DayOfWeek, and Store + DayOfWeek + Promo baselines across the same four monthly validation periods.

All historical statistics must be calculated using the training portion of each fold.

In [ ]:
def evaluate_group_mean_baseline(train_df, valid_df, group_cols):
    train_open = train_df.loc[train_df["Open"] == 1]

    historical_means = (
        train_open.groupby(group_cols)["Sales"]
        .mean()
        .reset_index(name="HistoricalMeanSales")
    )

    predictions = valid_df[["Store", "DayOfWeek", "Open", "Sales"]].merge(
        historical_means,
        on=group_cols,
        how="left",
        validate="many_to_one",
    )

    store_means = train_open.groupby("Store")["Sales"].mean()

    predictions["PredictedSales"] = predictions["HistoricalMeanSales"].fillna(
        predictions["Store"].map(store_means)
    )

    predictions.loc[
        predictions["Open"] == 0,
        "PredictedSales",
    ] = 0

    if predictions["PredictedSales"].isna().any():
        raise ValueError("Missing final predictions")

    errors = predictions["Sales"] - predictions["PredictedSales"]
    open_errors = errors.loc[predictions["Open"] == 1]

    return {
        "overall_mae": errors.abs().mean(),
        "overall_rmse": errors.pow(2).mean() ** 0.5,
        "open_mae": open_errors.abs().mean(),
        "open_rmse": open_errors.pow(2).mean() ** 0.5,
    }

In [ ]:
july_store_metrics = evaluate_group_mean_baseline(
    train_data,
    valid_data,
    ["Store"],
)

july_weekday_metrics = evaluate_group_mean_baseline(
    train_data,
    valid_data,
    ["Store", "DayOfWeek"],
)

july_comparison = pd.DataFrame(
    {
        "Baseline 1": july_store_metrics,
        "Baseline 2": july_weekday_metrics,
    }
)

july_comparison.round(2)

In [ ]:
comparison_results = []

baselines = {
    "Store": ["Store"],
    "Store + Weekday": ["Store", "DayOfWeek"],
}

for start_date in validation_months:
    end_date = start_date + pd.offsets.MonthBegin(1)

    fold_train = df.loc[df["Date"] < start_date]

    fold_valid = df.loc[(df["Date"] >= start_date) & (df["Date"] < end_date)]

    month = start_date.strftime("%Y-%m")

    for model_name, group_cols in baselines.items():
        metrics = evaluate_group_mean_baseline(
            fold_train,
            fold_valid,
            group_cols,
        )

        comparison_results.append(
            {
                "month": month,
                "baseline": model_name,
                **metrics,
            }
        )

    promo_metrics = evaluate_promo_baseline(
        fold_train,
        fold_valid,
    )

    comparison_results.append(
        {
            "month": month,
            "baseline": "Store + Weekday + Promo",
            **promo_metrics,
        }
    )

comparison_df = pd.DataFrame(comparison_results)

comparison_df.head()

In [ ]:
mae_comparison = comparison_df.pivot(
    index="month",
    columns="baseline",
    values="open_mae",
)

mae_comparison.round(2)

In [ ]:
rmse_comparison = comparison_df.pivot(
    index="month",
    columns="baseline",
    values="open_rmse",
)

rmse_comparison.round(2)

## 9. Final Baseline Comparison and Conclusions

### Baseline Models

Three historical mean baselines were evaluated:

1. **Store Mean:** Average sales per store.
2. **Store + DayOfWeek:** Average sales per store and weekday.
3. **Store + DayOfWeek + Promo:** Average sales per store, weekday, and promotion status.

All historical averages were calculated using training data only.

### Validation Strategy

Expanding-window validation was performed across April, May, June, and July 2015.

Each validation month was predicted using only historical sales observations from earlier dates.

### Key Results

The Store + DayOfWeek + Promo baseline achieved the lowest open-day MAE and RMSE in all four validation months.

For July 2015, its performance was:

- Open-day MAE: 732.18
- Open-day RMSE: 1,038.49
- Overall MAE: 639.46
- Overall RMSE: 970.52

Adding weekday information improved predictions over the Store Mean baseline, and incorporating promotion status provided further substantial improvements.

### Error Analysis

April showed higher prediction errors than the other validation months.

Several dates near Easter contributed substantially to the total absolute error, suggesting that additional calendar-related features may improve predictions.

### Modeling Decisions

- Select **Store + DayOfWeek + Promo** as the current reference baseline.
- Assume future Open and Promo information is available.
- Do not use future Customers or Sales as predictive features.
- Preserve legitimate high-sales observations.
- Consider holiday proximity and other calendar features in future models.
- Evaluate future models using chronological validation and a separately reserved final test period when available.

### Limitations

This baseline does not directly account for seasonal changes, holidays, competition, or other potentially useful features.

The validation months were used during model development, so their results should not be treated as a fully independent final performance estimate.

### Conclusion

A simple historical average baseline can provide useful sales forecasts when store identity, weekday, and promotion status are considered.

The selected baseline establishes a reproducible performance reference for future machine learning models.